# Clases y objetos · Polinomios

Una clase `Polinomio` inmutable con dos representaciones que comparten la misma interfaz: la *densa* guarda todos los coeficientes en una tupla, del término independiente al de mayor grado, y la *dispersa* solo los no nulos en un diccionario grado → coeficiente, mucho más pequeño para polinomios como $x^{1000} + 1$. La clase base solo necesita que cada representación sepa dar sus términos y construirse a partir de ellos; el resto de operaciones (evaluación con Horner, aritmética, derivada, división, Ruffini, raíces enteras, factorización, máximo común divisor y composición) se escriben una sola vez y valen para las dos. Los coeficientes son enteros o `Fraction`, así que todo es exacto, y los resultados se cruzan con `numpy.polynomial` y `sympy`. Ideas de los ejercicios de [José A. Alonso](https://github.com/jaalonso/Ejercicios-Python) y de [Edabit](https://edabit.com/challenges/python3).

## 1. Representación densa y dispersa

Crear una clase abstracta `Polinomio` con dos métodos abstractos, `terminos()` (diccionario grado → coeficiente sin ceros, de mayor a menor grado) y el constructor de clase `desde_terminos(terminos)`, y sobre ellos `grado` (−1 para el polinomio cero), `coeficiente(k)`, `principal` (el coeficiente principal), `es_cero`, `a_lista` (coeficientes de menor a mayor grado), `a_densa`, `a_dispersa`, `__eq__` (también con números) y `__hash__`. Implementarla con dos `dataclass` inmutables: `Densa`, con una tupla de coeficientes sin ceros finales, y `Dispersa`, con un diccionario de solo lectura (`MappingProxyType`). Comprobar que un polinomio es igual en ambas representaciones y que la conversión va y vuelve sin cambios.

In [1]:
import numbers
import operator
import random
from abc import ABC, abstractmethod
from collections.abc import Mapping
from dataclasses import FrozenInstanceError, dataclass, field
from fractions import Fraction
from math import comb, gcd, isqrt
from types import MappingProxyType

import numpy as np
import sympy
from numpy.polynomial import polynomial as P
from sympy.parsing.sympy_parser import (convert_xor, implicit_multiplication_application, parse_expr,
                                        standard_transformations)


class Polinomio(ABC):
    @abstractmethod
    def terminos(self) -> dict[int, object]:
        """Grado → coeficiente, sin ceros y de mayor a menor grado."""

    @classmethod
    @abstractmethod
    def desde_terminos(cls, terminos: Mapping[int, object]) -> "Polinomio":
        """Construye el polinomio; los términos pueden traer ceros y venir en cualquier orden."""

    @property
    def grado(self) -> int:
        return next(iter(self.terminos()), -1)

    def coeficiente(self, k: int):
        return self.terminos().get(k, 0)

    @property
    def principal(self):
        return self.coeficiente(self.grado)

    def es_cero(self) -> bool:
        return not self.terminos()

    def a_lista(self) -> list:
        return [self.coeficiente(k) for k in range(self.grado + 1)]

    def a_densa(self) -> "Densa":
        return Densa.desde_terminos(self.terminos())

    def a_dispersa(self) -> "Dispersa":
        return Dispersa.desde_terminos(self.terminos())

    def __eq__(self, otro):
        match otro:
            case Polinomio():
                return self.terminos() == otro.terminos()
            case numbers.Number():
                return self.terminos() == ({0: otro} if otro else {})
            case _:
                return NotImplemented

    def __hash__(self):
        return hash(tuple(self.terminos().items()))


@dataclass(frozen=True, eq=False)
class Densa(Polinomio):
    coefs: tuple = ()

    def __post_init__(self):
        c = list(self.coefs)
        while c and c[-1] == 0:
            c.pop()
        object.__setattr__(self, "coefs", tuple(c))

    def terminos(self):
        return {k: c for k, c in reversed(list(enumerate(self.coefs))) if c != 0}

    @classmethod
    def desde_terminos(cls, terminos):
        c = [0] * (max(terminos, default=-1) + 1)
        for k, v in terminos.items():
            c[k] += v
        return cls(tuple(c))


@dataclass(frozen=True, eq=False, repr=False)
class Dispersa(Polinomio):
    coefs: Mapping = field(default_factory=dict)

    def __post_init__(self):
        if any(not isinstance(k, int) or k < 0 for k in self.coefs):
            raise ValueError("los grados deben ser enteros no negativos")
        limpio = {k: self.coefs[k] for k in sorted(self.coefs, reverse=True) if self.coefs[k] != 0}
        object.__setattr__(self, "coefs", MappingProxyType(limpio))

    def terminos(self):
        return dict(self.coefs)

    @classmethod
    def desde_terminos(cls, terminos):
        return cls(terminos)

    def __repr__(self):
        return f"Dispersa({dict(self.coefs)})"


densa = Densa((2, -1, 0, 0, 3, 0, 0))
dispersa = Dispersa({4: 3, 1: -1, 0: 2, 7: 0})
print(densa, "|", dispersa)
print(f"grado {densa.grado}, principal {densa.principal}, coeficiente de x: {densa.coeficiente(1)}, lista: {dispersa.a_lista()}")
print("¿Iguales?", densa == dispersa, "| a_dispersa():", densa.a_dispersa(), "| a_densa():", dispersa.a_densa())
print("Cero:", Densa(), Dispersa(), "grado", Densa().grado, "| constante 5 == 5:", Densa((5,)) == 5)

grande_densa = Densa((1,) + (0,) * 999 + (1,))
grande_dispersa = grande_densa.a_dispersa()
print(f"\nx^1000 + 1: {len(grande_densa.coefs)} coeficientes en la densa, {len(grande_dispersa.coefs)} en la dispersa")

for prueba in (lambda: setattr(densa, "coefs", (1,)), lambda: operator.setitem(dispersa.coefs, 9, 1), lambda: Dispersa({-1: 2})):
    try:
        prueba()
    except (FrozenInstanceError, TypeError, ValueError) as error:
        print(f"{type(error).__name__}: {error}")

assert densa == dispersa and hash(densa) == hash(dispersa) and len({densa, dispersa}) == 1
assert densa.a_dispersa().a_densa() == densa and densa.coefs == (2, -1, 0, 0, 3)
assert Densa() == Dispersa() == 0 and Densa().grado == -1 and Densa().principal == 0
assert grande_densa == grande_dispersa and grande_dispersa.grado == 1000

Densa(coefs=(2, -1, 0, 0, 3)) | Dispersa({4: 3, 1: -1, 0: 2})
grado 4, principal 3, coeficiente de x: -1, lista: [2, -1, 0, 0, 3]
¿Iguales? True | a_dispersa(): Dispersa({4: 3, 1: -1, 0: 2}) | a_densa(): Densa(coefs=(2, -1, 0, 0, 3))
Cero: Densa(coefs=()) Dispersa({}) grado -1 | constante 5 == 5: True

x^1000 + 1: 1001 coeficientes en la densa, 2 en la dispersa
FrozenInstanceError: cannot assign to field 'coefs'
TypeError: 'mappingproxy' object does not support item assignment
ValueError: los grados deben ser enteros no negativos


## 2. Representación legible con `__str__`

Escribir `__str__` para que los polinomios se lean como en matemáticas: de mayor a menor grado, `3x^4 - x + 2`, sin coeficientes 1 (salvo en el término independiente), sin `^1`, con el signo del primer término pegado y las fracciones entre paréntesis cuando acompañan a la `x`: `(1/2)x^2`. El polinomio cero se escribe `0`. Para ir completando la clase ejercicio a ejercicio, cada método se añade a `Polinomio` con un pequeño decorador `metodo` y lo heredan las dos representaciones. Comprobar con polinomios aleatorios que `sympy` lee la cadena como el mismo polinomio.

In [2]:
def metodo(funcion):
    """Añade la función a Polinomio como método (en un módulo iría directamente en la clase)."""
    setattr(Polinomio, funcion.__name__, funcion)
    return funcion


def _monomio(c, k: int) -> str:
    if c == 1 and k > 0:
        coef = ""
    elif isinstance(c, Fraction) and c.denominator != 1 and k > 0:
        coef = f"({c})"
    else:
        coef = str(c)
    return coef + ("" if k == 0 else "x" if k == 1 else f"x^{k}")


@metodo
def __str__(self) -> str:
    texto = ""
    for k, c in self.terminos().items():
        m = _monomio(abs(c), k)
        if not texto:
            texto = "-" + m if c < 0 else m
        else:
            texto += (" - " if c < 0 else " + ") + m
    return texto or "0"


ejemplos = [Densa((2, -1, 0, 0, 3)), Dispersa({3: -1, 1: 1}), Densa((-7,)), Densa(), Dispersa({5: 1, 0: -1}),
            Densa((Fraction(-1, 3), 0, Fraction(1, 2), Fraction(4))), Dispersa({1: -1})]
for p in ejemplos:
    print(f"{p!r:<55} {p}")


def aleatorio(rng: random.Random, grado_max: int = 6, tipo=None, cmax: int = 5) -> Polinomio:
    tipo = tipo or rng.choice([Densa, Dispersa])
    return tipo.desde_terminos({k: rng.randint(-cmax, cmax) for k in range(rng.randint(0, grado_max) + 1)})


TRANSFORMACIONES = standard_transformations + (implicit_multiplication_application, convert_xor)
xs = sympy.Symbol("x")


def a_sympy(p: Polinomio):
    return sum((sympy.Rational(str(c)) * xs**k for k, c in p.terminos().items()), sympy.Integer(0))


rng = random.Random(2)
for _ in range(300):
    p = aleatorio(rng)
    if rng.random() < 0.3:
        p = Densa(tuple(Fraction(c, rng.randint(1, 4)) for c in p.a_lista()))
    assert sympy.expand(parse_expr(str(p), transformations=TRANSFORMACIONES) - a_sympy(p)) == 0, p
assert str(Densa((2, -1, 0, 0, 3))) == "3x^4 - x + 2"

Densa(coefs=(2, -1, 0, 0, 3))                           3x^4 - x + 2
Dispersa({3: -1, 1: 1})                                 -x^3 + x
Densa(coefs=(-7,))                                      -7
Densa(coefs=())                                         0
Dispersa({5: 1, 0: -1})                                 x^5 - 1
Densa(coefs=(Fraction(-1, 3), 0, Fraction(1, 2), Fraction(4, 1))) 4x^3 + (1/2)x^2 - 1/3
Dispersa({1: -1})                                       -x


## 3. Evaluación con el esquema de Horner

Hacer que los polinomios sean invocables, `p(x)`, con el esquema de Horner: $a_n x^n + \dots + a_1 x + a_0 = (\cdots((a_n x + a_{n-1}) x + a_{n-2}) \cdots) x + a_0$, que necesita unas $n$ multiplicaciones en lugar de las unas $n^2/2$ de calcular cada potencia por separado. En la dispersa, cuando faltan términos, se salta de un grado al siguiente multiplicando por $x^{\text{salto}}$. Contar las multiplicaciones de ambos métodos para varios grados y comprobar los valores con la evaluación directa, `numpy.polynomial.polynomial.polyval` y `numpy.polyval` (que espera los coeficientes de mayor a menor grado).

In [3]:
@metodo
def __call__(self, x):
    terminos = self.terminos()
    if not terminos:
        return x * 0
    acumulado, previo = 0, self.grado
    for k, c in terminos.items():
        acumulado = acumulado * x ** (previo - k) + c
        previo = k
    return acumulado * x**previo


def directa(p: Polinomio, x) -> tuple[object, int]:
    total, productos = 0, 0
    for k, c in p.terminos().items():
        potencia = 1
        for _ in range(k):
            potencia *= x
            productos += 1
        total += c * potencia
        productos += 1
    return total, productos


def horner(p: Polinomio, x) -> tuple[object, int]:
    acumulado, productos = 0, 0
    for k in range(p.grado, -1, -1):
        acumulado = acumulado * x + p.coeficiente(k)
        productos += 1
    return acumulado, productos


p = Densa((2, -1, 0, 0, 3))
print(f"p(x) = {p}:  p(2) = {p(2)},  p(1/2) = {p(Fraction(1, 2))},  p(-1.5) = {p(-1.5)}")
print(f"\n{'grado':>5} {'directa':>8} {'Horner':>7}")
rng = random.Random(3)
for n in (5, 10, 50, 100):
    q = Densa(tuple(rng.randint(1, 9) for _ in range(n + 1)))
    (v1, m1), (v2, m2) = directa(q, 3), horner(q, 3)
    assert v1 == v2 == q(3)
    print(f"{n:>5} {m1:>8} {m2:>7}")

for _ in range(300):
    q = aleatorio(rng, 10)
    t = Fraction(rng.randint(-9, 9), rng.randint(1, 5))
    assert q(t) == directa(q, t)[0] == horner(q, t)[0]
    coefs, real = [float(c) for c in q.a_lista()] or [0.0], float(t)
    assert np.isclose(q(real), P.polyval(real, coefs)) and np.isclose(q(real), np.polyval(coefs[::-1], real))
assert Dispersa({1000: 1, 0: 1})(2) == 2**1000 + 1

p(x) = 3x^4 - x + 2:  p(2) = 48,  p(1/2) = 27/16,  p(-1.5) = 18.6875

grado  directa  Horner
    5       21       6
   10       66      11
   50     1326      51
  100     5151     101


## 4. Suma, resta y producto

Añadir `__add__`, `__sub__`, `__neg__` y `__mul__`, con sus versiones por la derecha para operar con números (`2 * p`, `1 - p`). El resultado es del tipo del operando izquierdo, así que se pueden mezclar representaciones. El producto multiplica cada término por cada término y acumula por grados. Con `X = Densa((0, 1))` los polinomios ya se escriben de forma natural. Comprobar con polinomios aleatorios contra `polyadd`, `polysub` y `polymul` de `numpy.polynomial.polynomial`.

In [4]:
def _como_polinomio(p: Polinomio, otro) -> Polinomio | None:
    match otro:
        case Polinomio():
            return otro
        case numbers.Number():
            return type(p).desde_terminos({0: otro})
        case _:
            return None


@metodo
def __add__(self, otro):
    if (otro := _como_polinomio(self, otro)) is None:
        return NotImplemented
    t = self.terminos()
    for k, c in otro.terminos().items():
        t[k] = t.get(k, 0) + c
    return type(self).desde_terminos(t)


@metodo
def __neg__(self):
    return type(self).desde_terminos({k: -c for k, c in self.terminos().items()})


@metodo
def __sub__(self, otro):
    if (otro := _como_polinomio(self, otro)) is None:
        return NotImplemented
    return self + (-otro)


@metodo
def __rsub__(self, otro):
    return -self + otro


@metodo
def __mul__(self, otro):
    if (otro := _como_polinomio(self, otro)) is None:
        return NotImplemented
    t = {}
    for i, a in self.terminos().items():
        for j, b in otro.terminos().items():
            t[i + j] = t.get(i + j, 0) + a * b
    return type(self).desde_terminos(t)


Polinomio.__radd__ = Polinomio.__add__
Polinomio.__rmul__ = Polinomio.__mul__

X = Densa((0, 1))
p = 3 * X * X * X * X - X + 2
q = Dispersa({2: 1, 0: -1})
print(f"p = {p!r}  →  {p}")
print(f"p + q = {p + q}\np - q = {p - q}\nq - p = {q - p}  ({type(q - p).__name__})\np · q = {p * q}\n1 - q = {1 - q}")


def a_entero(arr) -> Densa:
    return Densa(tuple(int(v) for v in arr))


rng = random.Random(4)
for _ in range(500):
    a, b = aleatorio(rng), aleatorio(rng)
    la, lb = a.a_lista() or [0], b.a_lista() or [0]
    assert a + b == a_entero(P.polyadd(la, lb)) and a - b == a_entero(P.polysub(la, lb))
    assert a * b == a_entero(P.polymul(la, lb)) and a * b == b * a and (a - b) + b == a
    assert type(a + b) is type(a) and 2 * a == a + a and a - a == 0

p = Densa(coefs=(2, -1, 0, 0, 3))  →  3x^4 - x + 2
p + q = 3x^4 + x^2 - x + 1
p - q = 3x^4 - x^2 - x + 3
q - p = -3x^4 + x^2 + x - 3  (Dispersa)
p · q = 3x^6 - 3x^4 - x^3 + 2x^2 + x - 2
1 - q = -x^2 + 2


## 5. Potencia, grado y coeficiente principal

Añadir `__pow__` para exponentes naturales con exponenciación rápida (elevar al cuadrado y multiplicar según los bits del exponente). Comprobar que coincide con multiplicar el polinomio por sí mismo, que $(x + 1)^n$ tiene como coeficientes los números combinatorios, que el grado de $p^n$ es $n$ veces el de $p$ y su coeficiente principal la potencia del principal, y contra `polypow`. Ver también la ventaja de la dispersa al elevar $x^{1000} + 1$.

In [5]:
@metodo
def __pow__(self, n: int):
    if isinstance(n, bool) or not isinstance(n, int) or n < 0:
        raise ValueError("el exponente debe ser un entero no negativo")
    resultado, base = type(self).desde_terminos({0: 1}), self
    while n:
        if n & 1:
            resultado = resultado * base
        base = base * base
        n >>= 1
    return resultado


print(f"(x + 1)^6 = {(X + 1) ** 6}")
print(f"(2x - 3)^3 = {(2 * X - 3) ** 3}")
cubo_disperso = Dispersa({1000: 1, 0: 1}) ** 3
cubo_denso = Densa.desde_terminos({1000: 1, 0: 1}) ** 3
print(f"(x^1000 + 1)^3 = {cubo_disperso}  ({len(cubo_disperso.coefs)} términos guardados frente a {len(cubo_denso.coefs)})")
try:
    X ** -1
except ValueError as error:
    print("ValueError:", error)

for n in range(21):
    assert ((X + 1) ** n).a_lista() == [comb(n, k) for k in range(n + 1)]
rng = random.Random(5)
for _ in range(200):
    p, n = aleatorio(rng, 4), rng.randint(0, 6)
    producto = type(p).desde_terminos({0: 1})
    for _ in range(n):
        producto = producto * p
    assert p**n == producto
    if not p.es_cero():
        assert (p**n).grado == n * p.grado and (p**n).principal == p.principal**n
        assert p**n == a_entero(P.polypow(p.a_lista(), n))
assert cubo_disperso == cubo_denso

(x + 1)^6 = x^6 + 6x^5 + 15x^4 + 20x^3 + 15x^2 + 6x + 1
(2x - 3)^3 = 8x^3 - 36x^2 + 54x - 27
(x^1000 + 1)^3 = x^3000 + 3x^2000 + 3x^1000 + 1  (4 términos guardados frente a 3001)
ValueError: el exponente debe ser un entero no negativo


## 6. Derivada e integral

Añadir `derivada()`, `integral(constante=0)` (con coeficientes `Fraction` para que sea exacta) e `integral_definida(a, b)` con la regla de Barrow, $\int_a^b p(x)\,dx = P(b) - P(a)$. Comprobar que derivar la integral devuelve el polinomio, que integrar la derivada lo devuelve salvo la constante $p(0)$, y los resultados contra `polyder` y `polyint`.

In [6]:
@metodo
def derivada(self):
    return type(self).desde_terminos({k - 1: k * c for k, c in self.terminos().items() if k > 0})


@metodo
def integral(self, constante=0):
    t = {k + 1: Fraction(c) / (k + 1) for k, c in self.terminos().items()}
    t[0] = constante
    return type(self).desde_terminos(t)


@metodo
def integral_definida(self, a, b):
    primitiva = self.integral()
    return primitiva(b) - primitiva(a)


p = 3 * X**2 - 2 * X + 1
print(f"p = {p}\np' = {p.derivada()}\n∫p = {p.integral()}\n∫p desde 0 hasta 2 = {p.integral_definida(0, 2)}")
q = Dispersa({4: 5, 1: 1, 0: -3})
print(f"q = {q}\nq' = {q.derivada()},  q'' = {q.derivada().derivada()}\n∫q = {q.integral(7)}")

rng = random.Random(6)
for _ in range(300):
    p = aleatorio(rng, 8)
    assert p.integral().derivada() == p
    assert p.derivada().integral() == p - p(0)
    lista = p.a_lista() or [0]
    assert np.allclose([float(c) for c in p.derivada().a_lista()] or [0], P.polyder(lista))
    assert np.allclose([float(c) for c in p.integral().a_lista()] or [0], P.polyint(lista))
    a, b = rng.randint(-3, 3), rng.randint(-3, 3)
    assert p.integral_definida(a, b) == sympy.integrate(a_sympy(p), (xs, a, b))

p = 3x^2 - 2x + 1
p' = 6x - 2
∫p = x^3 - x^2 + x
∫p desde 0 hasta 2 = 6
q = 5x^4 + x - 3
q' = 20x^3 + 1,  q'' = 60x^2
∫q = x^5 + (1/2)x^2 - 3x + 7


## 7. División con resto

Añadir `__divmod__` (y con él `__floordiv__` y `__mod__`): mientras el grado del resto no baje del divisor, se divide el término principal del resto entre el del divisor, se añade ese monomio al cociente y se resta del resto el monomio por el divisor. Los coeficientes del cociente son `Fraction`, así que la división es exacta aunque el divisor no sea mónico. Comprobar que $p = c \cdot d + r$ con $\text{grado}(r) < \text{grado}(d)$, comparar con `polydiv` y mostrar el error de dividir entre el polinomio cero.

In [7]:
@metodo
def __divmod__(self, divisor):
    if (divisor := _como_polinomio(self, divisor)) is None:
        return NotImplemented
    if divisor.es_cero():
        raise ZeroDivisionError("división entre el polinomio cero")
    cociente, resto = {}, self
    while resto.grado >= divisor.grado:
        k = resto.grado - divisor.grado
        c = Fraction(resto.principal) / divisor.principal
        cociente[k] = c
        resto = resto - type(self).desde_terminos({k: c}) * divisor
    return type(self).desde_terminos(cociente), resto


@metodo
def __floordiv__(self, divisor):
    return divmod(self, divisor)[0]


@metodo
def __mod__(self, divisor):
    return divmod(self, divisor)[1]


for p, d in [(X**4 - 3 * X**2 + X + 5, X**2 + 1), (2 * X**3 + 3 * X - 1, 2 * X - 1), (X**3 - 1, X - 1), (X + 4, X**2)]:
    c, r = divmod(p, d)
    print(f"({p}) = ({d})·({c}) + ({r})")
try:
    divmod(X**2, Densa())
except ZeroDivisionError as error:
    print("ZeroDivisionError:", error)

rng = random.Random(7)
for _ in range(400):
    p, d = aleatorio(rng, 8), aleatorio(rng, 4)
    if d.es_cero():
        continue
    c, r = divmod(p, d)
    assert p == c * d + r and r.grado < d.grado
    c_np, r_np = P.polydiv([float(v) for v in p.a_lista()] or [0.0], [float(v) for v in d.a_lista()])
    assert np.allclose([float(v) for v in c.a_lista()] or [0], c_np)
    resto = [float(v) for v in r.a_lista()]
    assert np.allclose(resto + [0.0] * (len(r_np) - len(resto)), r_np)

(x^4 - 3x^2 + x + 5) = (x^2 + 1)·(x^2 - 4) + (x + 9)
(2x^3 + 3x - 1) = (2x - 1)·(x^2 + (1/2)x + 7/4) + (3/4)
(x^3 - 1) = (x - 1)·(x^2 + x + 1) + (0)
(x + 4) = (x^2)·(0) + (x + 4)
ZeroDivisionError: división entre el polinomio cero


## 8. Regla de Ruffini

Para dividir entre $x - a$ basta la regla de Ruffini: se baja el primer coeficiente y cada siguiente es el coeficiente original más $a$ por el último obtenido; el último número es el resto. Escribir `ruffini(p, a, mostrar=False)`, que devuelva el cociente y el resto y, si se pide, dibuje la tabla como en el instituto. Comprobar el teorema del resto (el resto es $p(a)$) y que coincide con `divmod(p, X - a)`.

In [8]:
def ruffini(p: Polinomio, a: int, mostrar: bool = False) -> tuple[Polinomio, object]:
    coefs = p.a_lista()[::-1]
    fila, productos = coefs[:1], []
    for c in coefs[1:]:
        productos.append(a * fila[-1])
        fila.append(c + productos[-1])
    if mostrar:
        w = max(len(str(v)) for v in coefs + productos + fila) + 3
        print(" " * 6 + "|" + "".join(f"{str(c):>{w}}" for c in coefs))
        print(f"{str(a):>5} |" + " " * w + "".join(f"{str(v):>{w}}" for v in productos))
        print(" " * 6 + "+" + "-" * (w * len(coefs)))
        print(" " * 7 + "".join(f"{str(v):>{w}}" for v in fila[:-1]) + f" | {str(fila[-1]):>{w - 3}}")
    cociente = type(p).desde_terminos(dict(enumerate(reversed(fila[:-1]))))
    return cociente, fila[-1] if fila else 0


for p, a in [(X**3 - 7 * X + 6, 2), (2 * X**4 - 3 * X**2 + X - 5, -1)]:
    print(f"({p}) : (x - ({a}))")
    c, r = ruffini(p, a, mostrar=True)
    print(f"cociente {c}, resto {r}, p({a}) = {p(a)}\n")

rng = random.Random(8)
for _ in range(400):
    p, a = aleatorio(rng, 8), rng.randint(-5, 5)
    c, r = ruffini(p, a)
    assert r == p(a) and (c, r) == divmod(p, X - a) and p == c * (X - a) + r

(x^3 - 7x + 6) : (x - (2))
      |    1    0   -7    6
    2 |         2    4   -6
      +--------------------
           1    2   -3 |  0
cociente x^2 + 2x - 3, resto 0, p(2) = 0

(2x^4 - 3x^2 + x - 5) : (x - (-1))
      |    2    0   -3    1   -5
   -1 |        -2    2    1   -2
      +-------------------------
           2   -2   -1    2 | -7
cociente 2x^3 - 2x^2 - x + 2, resto -7, p(-1) = -7



## 9. Raíces enteras

Si un polinomio de coeficientes enteros tiene una raíz entera $r \neq 0$, $r$ divide al término independiente. Escribir `raices_enteras(p)`: si el término independiente es 0, $x = 0$ es raíz y se sigue con el primer coeficiente no nulo; después se prueban solo los divisores positivos y negativos de ese coeficiente. Comparar cuántos candidatos prueba con la fuerza bruta (todos los enteros de $-|a_0|$ a $|a_0|$) y comprobar que encuentran las mismas raíces.

In [9]:
def divisores(n: int) -> list[int]:
    n = abs(n)
    return sorted({d for i in range(1, isqrt(n) + 1) if n % i == 0 for d in (i, n // i)})


def raices_enteras(p: Polinomio) -> list[int]:
    if p.es_cero():
        raise ValueError("el polinomio cero se anula en todos los puntos")
    if any(Fraction(c).denominator != 1 for c in p.terminos().values()):
        raise ValueError("los coeficientes deben ser enteros")
    p = type(p).desde_terminos({k: int(c) for k, c in p.terminos().items()})
    menor = min(p.terminos())
    candidatos = [s * d for d in divisores(p.coeficiente(menor)) for s in (1, -1)]
    return sorted([0] * (menor > 0) + [r for r in candidatos if p(r) == 0])


def raices_bruta(p: Polinomio) -> list[int]:
    cota = abs(p.coeficiente(min(p.terminos())))
    return [r for r in range(-cota, cota + 1) if p(r) == 0]


p = (X - 3) * (X + 2) * (X - 5) * (X**2 + 1)
print(f"p = {p}")
print(f"raíces enteras: {raices_enteras(p)}; candidatos probados: {2 * len(divisores(p.coeficiente(0)))} "
      f"frente a {2 * abs(p.coeficiente(0)) + 1} con fuerza bruta")
print(f"raíces de {X**3 - 4 * X}: {raices_enteras(X**3 - 4 * X)};  de {X**2 + X + 1}: {raices_enteras(X**2 + X + 1)}")
try:
    raices_enteras(X - Fraction(1, 2))
except ValueError as error:
    print("ValueError:", error)

rng = random.Random(9)
for _ in range(300):
    raices = [rng.randint(-6, 6) for _ in range(rng.randint(0, 3))]
    p = aleatorio(rng, 3, cmax=4)
    if p.es_cero():
        continue
    for r in raices:
        p = p * (X - r)
    assert raices_enteras(p) == raices_bruta(p) and set(raices) <= set(raices_enteras(p))

p = x^5 - 6x^4 + 24x^2 - x + 30
raíces enteras: [-2, 3, 5]; candidatos probados: 16 frente a 61 con fuerza bruta
raíces de x^3 - 4x: [-2, 0, 2];  de x^2 + x + 1: []
ValueError: los coeficientes deben ser enteros


## 10. Factorización con raíces enteras

Factorizar un polinomio de coeficientes enteros como una constante (el máximo común divisor de los coeficientes, con el signo del principal), factores lineales $(x - r)^m$ con sus raíces enteras y un factor restante sin raíces enteras: para cada raíz se aplica Ruffini mientras el resto sea 0, contando la multiplicidad. Escribir `factorizar(p)`, que devuelva la constante, la lista de `(raíz, multiplicidad)` y el factor restante, y `texto_factorizacion`, que la escriba como `2(x - 1)^2(x + 3)(x^2 + 1)`. Comprobar que el producto de los factores devuelve el polinomio y que el factor restante ya no tiene raíces enteras.

In [10]:
def factorizar(p: Polinomio) -> tuple[int, list[tuple[int, int]], Polinomio]:
    raices = raices_enteras(p)
    constante = gcd(*(int(c) for c in p.terminos().values())) * (1 if p.principal > 0 else -1)
    resto = type(p).desde_terminos({k: int(c) // constante for k, c in p.terminos().items()})
    factores = []
    for r in raices:
        m = 0
        while True:
            cociente, residuo = ruffini(resto, r)
            if residuo != 0:
                break
            resto, m = cociente, m + 1
        factores.append((r, m))
    return constante, factores, resto


def texto_factorizacion(constante: int, factores, resto: Polinomio) -> str:
    partes = []
    for r, m in factores:
        base = "x" if r == 0 else f"(x - {r})" if r > 0 else f"(x + {-r})"
        partes.append(base + (f"^{m}" if m > 1 else ""))
    if resto.grado > 0:
        partes.append(f"({resto})")
    if not partes:
        return str(constante)
    return {1: "", -1: "-"}.get(constante, str(constante)) + "".join(partes)


for p in [2 * (X - 1) ** 2 * (X + 3) * (X**2 + 1), X**5 - X**3, -6 * X**2 + 6, Densa((12,)), -(X**3) + 3 * X - 2, 3 * X**2 + 3, X**4 - 5 * X**2 + 4]:
    print(f"{str(p):<34} = {texto_factorizacion(*factorizar(p))}")

rng = random.Random(10)
for _ in range(300):
    p = aleatorio(rng, 3, cmax=4)
    if p.es_cero():
        continue
    for _ in range(rng.randint(0, 4)):
        p = p * (X - rng.randint(-4, 4))
    constante, factores, resto = factorizar(p)
    reconstruido = constante * resto
    for r, m in factores:
        reconstruido = reconstruido * (X - r) ** m
    assert reconstruido == p and (resto == 1 or resto.grado > 0 and raices_enteras(resto) == [])
    assert resto.principal > 0 and gcd(*resto.terminos().values()) == 1
    assert sum(m for _, m in factores) + max(resto.grado, 0) == p.grado

2x^5 + 2x^4 - 8x^3 + 8x^2 - 10x + 6 = 2(x + 3)(x - 1)^2(x^2 + 1)
x^5 - x^3                          = (x + 1)x^3(x - 1)
-6x^2 + 6                          = -6(x + 1)(x - 1)
12                                 = 12
-x^3 + 3x - 2                      = -(x + 2)(x - 1)^2
3x^2 + 3                           = 3(x^2 + 1)
x^4 - 5x^2 + 4                     = (x + 2)(x + 1)(x - 1)(x - 2)


## 11. Máximo común divisor

Calcular el máximo común divisor de dos polinomios con el algoritmo de Euclides, $\text{mcd}(a, b) = \text{mcd}(b, a \bmod b)$, devolviéndolo mónico (coeficiente principal 1) con `Fraction`, ya que con enteros la división no siempre es exacta. Añadir `monico()` a la clase y escribir también el mínimo común múltiplo. Comprobar que el mcd divide a ambos, que al dividirlos por él quedan primos entre sí y que coincide con `sympy.gcd`.

In [11]:
@metodo
def monico(self):
    if self.es_cero():
        return self
    c = Fraction(self.principal)
    return type(self).desde_terminos({k: Fraction(v) / c for k, v in self.terminos().items()})


def mcd(a: Polinomio, b: Polinomio) -> Polinomio:
    while not b.es_cero():
        a, b = b, a % b
    return a.monico()


def mcm(a: Polinomio, b: Polinomio) -> Polinomio:
    return (a * b // mcd(a, b)).monico()


a = (X - 1) * (X + 2) ** 2 * (X**2 + 1)
b = (X + 2) * (X - 3) * (X**2 + 1)
print(f"a = {a}\nb = {b}\nmcd = {mcd(a, b)}  →  {texto_factorizacion(*factorizar(mcd(a, b)))}")
print(f"mcm = {mcm(a, b)}")
print(f"mcd({(2 * X - 1) * (X**2 + X + 1)}, {(2 * X - 1) * (X + 5)}) = {mcd((2 * X - 1) * (X**2 + X + 1), (2 * X - 1) * (X + 5))}")

rng = random.Random(11)
for _ in range(150):
    comun = aleatorio(rng, 3, cmax=3)
    a, b = aleatorio(rng, 3, cmax=3) * comun, aleatorio(rng, 3, cmax=3) * comun
    if a.es_cero() or b.es_cero():
        continue
    g = mcd(a, b)
    assert (a % g).es_cero() and (b % g).es_cero() and mcd(a // g, b // g) == 1
    assert (comun.es_cero() or (g % comun.monico()).es_cero())
    esperado = sympy.Poly(sympy.gcd(a_sympy(a), a_sympy(b)), xs).monic()
    assert sympy.expand(a_sympy(g) - esperado.as_expr()) == 0
    assert mcm(a, b) * mcd(a, b) == (a * b).monico()

a = x^5 + 3x^4 + x^3 - x^2 - 4
b = x^4 - x^3 - 5x^2 - x - 6
mcd = x^3 + 2x^2 + x + 2  →  (x + 2)(x^2 + 1)
mcm = x^6 - 8x^4 - 4x^3 + 3x^2 - 4x + 12
mcd(2x^3 + x^2 + x - 1, 2x^2 + 9x - 5) = x - 1/2


## 12. Composición

Como el esquema de Horner solo usa sumas y productos, `p(q)` con `q` polinomio ya devuelve la composición $p(q(x))$ sin escribir nada nuevo. Mostrar $p \circ q$ y $q \circ p$ para dos polinomios y comprobar con polinomios aleatorios que $(p \circ q)(t) = p(q(t))$, que el grado es el producto de los grados, que la composición es asociativa y la regla de la cadena, $(p \circ q)' = p'(q) \cdot q'$.

In [12]:
p, q = X**2 + 1, X - 3
print(f"p = {p},  q = {q}\np(q(x)) = {p(q)}\nq(p(x)) = {q(p)}\np(p(x)) = {p(p)}")
print(f"(x^1000 + 1)(x^2) = {Dispersa({1000: 1, 0: 1})(Dispersa({2: 1}))}")

rng = random.Random(12)
for _ in range(200):
    p, q, r = aleatorio(rng, 4, cmax=3), aleatorio(rng, 3, cmax=3), aleatorio(rng, 2, cmax=3)
    pq = p(q)
    for t in range(-3, 4):
        assert pq(t) == p(q(t))
    if not p.es_cero() and q.grado > 0:
        assert pq.grado == p.grado * q.grado
    assert p(q(r)) == pq(r)
    assert pq.derivada() == p.derivada()(q) * q.derivada()

p = x^2 + 1,  q = x - 3
p(q(x)) = x^2 - 6x + 10
q(p(x)) = x^2 - 2
p(p(x)) = x^4 + 2x^2 + 2
(x^1000 + 1)(x^2) = x^2000 + 1


## 13. Propiedades algebraicas con polinomios aleatorios

Comprobar en una tabla, con 300 tríos de polinomios aleatorios (semilla fija, mezclando las dos representaciones), las propiedades conmutativa, asociativa y distributiva, el grado del producto, la regla del producto para la derivada, la división euclídea, que densa y dispersa dan el mismo resultado en todas las operaciones y que el producto coincide con el de la clase `numpy.polynomial.Polynomial`.

In [13]:
from numpy.polynomial import Polynomial

propiedades = {
    "p + q = q + p": lambda p, q, r: p + q == q + p,
    "(p + q) + r = p + (q + r)": lambda p, q, r: (p + q) + r == p + (q + r),
    "p·q = q·p": lambda p, q, r: p * q == q * p,
    "(p·q)·r = p·(q·r)": lambda p, q, r: (p * q) * r == p * (q * r),
    "p·(q + r) = p·q + p·r": lambda p, q, r: p * (q + r) == p * q + p * r,
    "grado(p·q) = grado p + grado q": lambda p, q, r: p.es_cero() or q.es_cero() or (p * q).grado == p.grado + q.grado,
    "(p·q)' = p'·q + p·q'": lambda p, q, r: (p * q).derivada() == p.derivada() * q + p * q.derivada(),
    "p = (p // q)·q + p % q": lambda p, q, r: q.es_cero() or p == (p // q) * q + p % q,
    "densa ≡ dispersa": lambda p, q, r: all(
        f(p.a_densa(), q.a_densa()).terminos() == f(p.a_dispersa(), q.a_dispersa()).terminos()
        for f in (lambda a, b: a + b, lambda a, b: a * b, lambda a, b: a(b), lambda a, b: a.derivada() - b ** 2)),
    "producto = numpy Polynomial": lambda p, q, r: np.array_equal(
        np.array((p * q).a_lista() or [0], dtype=float),
        (Polynomial(p.a_lista() or [0]) * Polynomial(q.a_lista() or [0])).trim().coef),
}

rng = random.Random(13)
casos = [tuple(aleatorio(rng, 6) for _ in range(3)) for _ in range(300)]
print(f"{'propiedad':<34} {'casos':>5} {'fallos':>6}")
for nombre, prop in propiedades.items():
    fallos = sum(not prop(*trio) for trio in casos)
    print(f"{nombre:<34} {len(casos):>5} {fallos:>6}")
    assert fallos == 0
print("Representaciones en los casos:", {t.__name__: sum(isinstance(p, t) for trio in casos for p in trio) for t in (Densa, Dispersa)})

propiedad                          casos fallos
p + q = q + p                        300      0
(p + q) + r = p + (q + r)            300      0
p·q = q·p                            300      0
(p·q)·r = p·(q·r)                    300      0
p·(q + r) = p·q + p·r                300      0
grado(p·q) = grado p + grado q       300      0
(p·q)' = p'·q + p·q'                 300      0
p = (p // q)·q + p % q               300      0


densa ≡ dispersa                     300      0
producto = numpy Polynomial          300      0
Representaciones en los casos: {'Densa': 458, 'Dispersa': 442}
